<a href="https://colab.research.google.com/github/Chanx-Charitana/LAB-1/blob/main/Henry_M__M_PG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. INSTALL & IMPORT

In [54]:
# ============================================================
# 0. INSTALL & IMPORT
# ============================================================
!pip install scikit-learn xgboost matplotlib seaborn pandas numpy shap openpyxl chardet

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import os
import gc
import warnings
import chardet
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive")

from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
import os
import pandas as pd

# ============================================================
# 🔧 PATHS — EDIT THESE
# ============================================================
BASE_PATH = "/content/drive/MyDrive/Biomass_Estimation_Project/01_RawData/data"

# Use only Excel now
EXCEL_PATH = os.path.join(BASE_PATH, "/content/drive/MyDrive/Biomass_Estimation_Project/01_RawData/data/data.csv")

# Output folder
OUTPUT_PATH = os.path.join(BASE_PATH, "Biomass_Estimation_Outputs")
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ============================================================
# 📂 LOAD DATA FROM EXCEL
# ============================================================

# Option 1: If ALL data is in ONE SHEET
df = pd.read_csv(EXCEL_PATH)

# 🔍 CHECK DATA
# ============================================================
print("Data loaded successfully!")
print(df.head())
print(df.columns)

Data loaded successfully!
  PLANT_ID;ndvi_mean;gndvi_mean;ndre_mean;msavi_mean;evi_mean;osavi_mean;vari_mean;arvi_mean;ndvi_std;ndre_std;evi_std;canopy_cover;chm_max;chm_mean;chm_std;volume;actual_biomass
0  3;0.36;0.4;0.16;0.14;0.16;0.21;0.58;0.21;0.09;...                                                                                                                              
1  5;0.47;0.46;0.2;0.21;0.24;0.28;0.58;0.35;0.16;...                                                                                                                              
2  7;0.44;0.44;0.19;0.18;0.2;0.26;0.57;0.3;0.12;0...                                                                                                                              
3  11;0.57;0.5;0.24;0.32;0.35;0.37;0.69;0.47;0.21...                                                                                                                              
4  12;0.48;0.46;0.21;0.21;0.24;0.29;0.58;0.35;0.1...                           

In [56]:
# 2. HELPER — SAFE CSV LOADER
# ============================================================
def load_csv_safe(path, label="file"):
    print(f"   Loading {label}...")
    encodings = ["utf-8", "utf-16", "latin-1", "cp1252", "iso-8859-1"]
    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"   ✅ {label} loaded ({enc}) — shape: {df.shape}")
            return df
        except Exception:
            continue
    with open(path, "rb") as f:
        detected = chardet.detect(f.read(50000))["encoding"]
    df = pd.read_csv(path, encoding=detected)
    print(f"   ✅ {label} loaded ({detected} detected) — shape: {df.shape}")
    return df


In [57]:
# ============================================================
# 3. LOAD DATA FROM EXCEL (COMBINED DATASET)
# ============================================================
print("📂 Loading biomass estimation data...\n")

# Define USE_EXCEL for this block to run, assuming intent to load data
USE_EXCEL = True

if USE_EXCEL:
    # Use the pre-defined EXCEL_PATH variable
    print(f"   Source: {EXCEL_PATH}")

    # Change to pd.read_csv since the file is a CSV and use the EXCEL_PATH variable
    df = pd.read_csv(EXCEL_PATH, sep=';')

    # Clean column names
    df.columns = df.columns.str.strip()

    print(f"\n   ✅ Dataset shape: {df.shape}")
    print(f"   📊 Columns: {list(df.columns)}")

    # 🔧 Set your target column (IMPORTANT)
    TARGET_COLUMN = "actual_biomass"   # ← change if needed

    # Split features and target
    X = df.drop(columns=[TARGET_COLUMN])
    y = df[TARGET_COLUMN]

    print(f"\n   🎯 Target column: {TARGET_COLUMN}")
    print(f"   📊 Number of features: {X.shape[1]}")
    print(f"   🌿 Features shape: {X.shape}")
    print(f"   🌳 Target shape  : {y.shape}")

else:
    raise ValueError("❌ CSV mode disabled — using combined Excel only.")

📂 Loading biomass estimation data...

   Source: /content/drive/MyDrive/Biomass_Estimation_Project/01_RawData/data/data.csv

   ✅ Dataset shape: (708, 18)
   📊 Columns: ['PLANT_ID', 'ndvi_mean', 'gndvi_mean', 'ndre_mean', 'msavi_mean', 'evi_mean', 'osavi_mean', 'vari_mean', 'arvi_mean', 'ndvi_std', 'ndre_std', 'evi_std', 'canopy_cover', 'chm_max', 'chm_mean', 'chm_std', 'volume', 'actual_biomass']

   🎯 Target column: actual_biomass
   📊 Number of features: 17
   🌿 Features shape: (708, 17)
   🌳 Target shape  : (708,)


In [58]:
# ============================================================
# 4. 🔧 COLUMN NAME MAP — COMBINED DATASET
# ============================================================

# 🎯 Target column (already confirmed from your previous code)
TARGET_COLUMN = "actual_biomass"   # ← your real biomass column

# 🧬 Feature columns (your ~10 features)
FEATURE_COLUMNS = [
      "volume",
    "chm_max",
    "chm_mean",
    "canopy_cover",
    "vari_mean",
    "gndvi_mean",
    "ndvi_mean",
    "ndvi_std",
    "ndre_std",
    "chm_std"
    # 👉 Add the rest of your features here if you have more
]

# ============================================================
# 📊 SELECT FEATURES & TARGET
# ============================================================

X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

print("\n📊 Selected Features:")
print(FEATURE_COLUMNS)

print(f"\n🎯 Target: {TARGET_COLUMN}")
print(f"🌿 Features shape: {X.shape}")
print(f"🌳 Target shape  : {y.shape}")


📊 Selected Features:
['volume', 'chm_max', 'chm_mean', 'canopy_cover', 'vari_mean', 'gndvi_mean', 'ndvi_mean', 'ndvi_std', 'ndre_std', 'chm_std']

🎯 Target: actual_biomass
🌿 Features shape: (708, 10)
🌳 Target shape  : (708,)


In [59]:
# ============================================================
# 5. PREVIEW DATA (COMBINED DATASET)
# ============================================================
print("\n📋 Data Preview:")

# Full dataset
print(f"\n── Full Dataset ──\n{df.head(3)}")

# Features
print(f"\n── Features (X) ──\n{X.head(3)}")

# Target
print(f"\n── Target (Actual Biomass) ──\n{y.head(3)}")



📋 Data Preview:

── Full Dataset ──
   PLANT_ID  ndvi_mean  gndvi_mean  ndre_mean  msavi_mean  evi_mean  \
0         3       0.36        0.40       0.16        0.14      0.16   
1         5       0.47        0.46       0.20        0.21      0.24   
2         7       0.44        0.44       0.19        0.18      0.20   

   osavi_mean  vari_mean  arvi_mean  ndvi_std  ndre_std  evi_std  \
0        0.21       0.58       0.21      0.09      0.16     0.07   
1        0.28       0.58       0.35      0.16      0.20     0.16   
2        0.26       0.57       0.30      0.12      0.19     0.11   

   canopy_cover  chm_max  chm_mean  chm_std  volume  actual_biomass  
0          0.00     0.20     0.010    0.033   0.010           31.27  
1          0.02     0.35     0.011    0.043   0.010           28.74  
2          0.00     0.18     0.018    0.034   0.018           14.70  

── Features (X) ──
   volume  chm_max  chm_mean  canopy_cover  vari_mean  gndvi_mean  ndvi_mean  \
0   0.010     0.20     0.

In [60]:
# ============================================================
# 7. FINAL DATASET (NO MERGE NEEDED — ALREADY COMBINED)
# ============================================================
print("\n🔗 Dataset already combined — no merging required.")

print(f"\n   ✅ Final dataset shape : {df.shape}")
print(f"   Columns              : {list(df.columns)}")
print(f"\n{df.head()}")

# ============================================================
# 💾 SAVE CLEAN DATASET
# ============================================================

final_path = os.path.join(OUTPUT_PATH, "final_biomass_dataset.csv")
df.to_csv(final_path, index=False, encoding="utf-8")

print(f"\n💾 Final dataset saved: {final_path}")


🔗 Dataset already combined — no merging required.

   ✅ Final dataset shape : (708, 18)
   Columns              : ['PLANT_ID', 'ndvi_mean', 'gndvi_mean', 'ndre_mean', 'msavi_mean', 'evi_mean', 'osavi_mean', 'vari_mean', 'arvi_mean', 'ndvi_std', 'ndre_std', 'evi_std', 'canopy_cover', 'chm_max', 'chm_mean', 'chm_std', 'volume', 'actual_biomass']

   PLANT_ID  ndvi_mean  gndvi_mean  ndre_mean  msavi_mean  evi_mean  \
0         3       0.36        0.40       0.16        0.14      0.16   
1         5       0.47        0.46       0.20        0.21      0.24   
2         7       0.44        0.44       0.19        0.18      0.20   
3        11       0.57        0.50       0.24        0.32      0.35   
4        12       0.48        0.46       0.21        0.21      0.24   

   osavi_mean  vari_mean  arvi_mean  ndvi_std  ndre_std  evi_std  \
0        0.21       0.58       0.21      0.09      0.16     0.07   
1        0.28       0.58       0.35      0.16      0.20     0.16   
2        0.26       0

In [61]:
# ============================================================
# 8. CLEAN & VALIDATE (COMBINED DATASET)
# ============================================================

# 🎯 Target column (must match exactly)
TARGET = "actual_biomass"

# 🌿 Feature columns (update if you have more)
FEATURES = [
       "volume",
    "chm_max",
    "chm_mean",
    "canopy_cover",
    "vari_mean",
    "gndvi_mean",
    "ndvi_mean",
    "ndvi_std",
    "ndre_std",
    "chm_std"
    # 👉 add your remaining features here if needed
]

print(f"\n   Features ({len(FEATURES)}) : {FEATURES}")
print(f"   Target                    : {TARGET}")

# ============================================================
# ✅ CHECK MISSING COLUMNS
# ============================================================
missing = [c for c in FEATURES + [TARGET] if c not in df.columns]

if missing:
    print(f"\n❌ Missing columns: {missing}")
    print(f"   Available columns: {list(df.columns)}")
    raise KeyError("Fix feature/target names to match your dataset.")
else:
    print(f"\n   ✅ All required columns found.")

# ============================================================
# 🧹 CLEAN DATA (REMOVE NaNs)
# ============================================================
before = len(df)

df = df.dropna(subset=FEATURES + [TARGET])

after = len(df)

print(f"\n   Rows before dropna : {before}")
print(f"   Rows after  dropna : {after} (dropped {before - after})")

# ============================================================
# 📊 TARGET STATISTICS
# ============================================================
print(f"\n   🌳 Actual Biomass statistics:\n{df[TARGET].describe()}")


   Features (10) : ['volume', 'chm_max', 'chm_mean', 'canopy_cover', 'vari_mean', 'gndvi_mean', 'ndvi_mean', 'ndvi_std', 'ndre_std', 'chm_std']
   Target                    : actual_biomass

   ✅ All required columns found.

   Rows before dropna : 708
   Rows after  dropna : 708 (dropped 0)

   🌳 Actual Biomass statistics:
count    708.000000
mean      67.010890
std       44.317526
min        0.930000
25%       37.415000
50%       61.240000
75%       89.215000
max      700.890000
Name: actual_biomass, dtype: float64


In [62]:
# ============================================================
# 9. FEATURES & TARGET — 70% TRAIN / 30% TEST
# ============================================================

# Use DataFrame directly (no need for .values unless required)
X = df[FEATURES]
y = df[TARGET]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print(f"\n   Total samples   : {len(X)}")
print(f"   Training samples: {len(X_train)} (70%) ← model training")
print(f"   Testing samples : {len(X_test)}  (30%) ← model evaluation")

# ============================================================
# 🔧 FEATURE SCALING (IMPORTANT for some models like SVM)
# ============================================================

scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("\n   ✅ Feature scaling applied (StandardScaler)")


   Total samples   : 708
   Training samples: 495 (70%) ← model training
   Testing samples : 213  (30%) ← model evaluation

   ✅ Feature scaling applied (StandardScaler)


In [63]:
# ============================================================
# 10. DEFINE BIOMASS ESTIMATION MODELS
# ============================================================

models = {
    "Support Vector Machine (SVM)": SVR(
        kernel="rbf",
        C=100,
        gamma=0.1,
        epsilon=0.1
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_split=4,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    )
}

print("\n🤖 Models defined:")
for name in models:
    print(f"   • {name}")


🤖 Models defined:
   • Support Vector Machine (SVM)
   • Random Forest
   • XGBoost


In [64]:
# ============================================================
# 11. TRAIN & EVALUATE BIOMASS ESTIMATION MODELS
# ============================================================

print("\n🌿 Training Biomass Estimation Models...")
print(f"   Features : {FEATURES}")
print(f"   Target   : {TARGET}")
print(f"   Split    : 70% Training | 30% Testing\n")

results     = {}
estimations = {}
cv_scores   = {}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"   ── {name} ──")

    # ✅ Use scaled data ONLY for SVM
    if "SVM" in name:
        Xtr, Xte, Xcv = X_train_sc, X_test_sc, X_train_sc
    else:
        Xtr, Xte, Xcv = X_train, X_test, X_train

    # ========================================================
    # 🚀 TRAIN MODEL
    # ========================================================
    model.fit(Xtr, y_train)

    # ========================================================
    # 🌳 PREDICT BIOMASS
    # ========================================================
    y_pred = model.predict(Xte)
    estimations[name] = y_pred

    # ========================================================
    # 📊 EVALUATION METRICS
    # ========================================================
    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)

    # Cross-validation (on training set)
    cv = cross_val_score(
        model, Xcv, y_train,
        cv=kf, scoring="r2", n_jobs=-1
    )

    cv_scores[name] = cv

    results[name] = {
        "R²"         : r2,
        "RMSE"       : rmse,
        "MAE"        : mae,
        "CV_R²_mean" : cv.mean(),
        "CV_R²_std"  : cv.std()
    }

    # ========================================================
    # 🖨️ PRINT RESULTS
    # ========================================================
    print(f"      R²   (Coefficient of Determination) : {r2:.4f}")
    print(f"      RMSE (Root Mean Square Error)       : {rmse:.4f}")
    print(f"      MAE  (Mean Absolute Error)          : {mae:.4f}")
    print(f"      CV R² (5-Fold)                      : {cv.mean():.4f} ± {cv.std():.4f}\n")


🌿 Training Biomass Estimation Models...
   Features : ['volume', 'chm_max', 'chm_mean', 'canopy_cover', 'vari_mean', 'gndvi_mean', 'ndvi_mean', 'ndvi_std', 'ndre_std', 'chm_std']
   Target   : actual_biomass
   Split    : 70% Training | 30% Testing

   ── Support Vector Machine (SVM) ──
      R²   (Coefficient of Determination) : 0.5248
      RMSE (Root Mean Square Error)       : 26.0398
      MAE  (Mean Absolute Error)          : 20.6443
      CV R² (5-Fold)                      : 0.2478 ± 0.0896

   ── Random Forest ──
      R²   (Coefficient of Determination) : 0.5017
      RMSE (Root Mean Square Error)       : 26.6664
      MAE  (Mean Absolute Error)          : 20.9973
      CV R² (5-Fold)                      : 0.0823 ± 0.2636

   ── XGBoost ──
      R²   (Coefficient of Determination) : 0.4792
      RMSE (Root Mean Square Error)       : 27.2627
      MAE  (Mean Absolute Error)          : 21.8044
      CV R² (5-Fold)                      : -0.0073 ± 0.3695



In [65]:
# 12. METRICS TABLE
# ============================================================
metrics_df = pd.DataFrame(results).T.round(4)
metrics_df.index.name = "Estimation Model"
metrics_df.to_csv(os.path.join(OUTPUT_PATH, "biomass_estimation_metrics.csv"))
print(f"📊 Biomass Estimation Metrics Summary:\n{metrics_df}\n")



📊 Biomass Estimation Metrics Summary:
                                  R²     RMSE      MAE  CV_R²_mean  CV_R²_std
Estimation Model                                                             
Support Vector Machine (SVM)  0.5248  26.0398  20.6443      0.2478     0.0896
Random Forest                 0.5017  26.6664  20.9973      0.0823     0.2636
XGBoost                       0.4792  27.2627  21.8044     -0.0073     0.3695



In [66]:
# 13. PLOT 1 — METRICS COMPARISON
# ============================================================
print("🖼️  Plot 1: Estimation metrics comparison...")

fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=80)
metric_cfg = [
    ("R²",   "#2196F3", "Coefficient of Determination\n(higher = better)"),
    ("RMSE", "#E53935", "Root Mean Square Error\n(lower = better)"),
    ("MAE",  "#FB8C00", "Mean Absolute Error\n(lower = better)"),
]

for ax, (metric, color, subtitle) in zip(axes, metric_cfg):
    vals  = [results[m][metric] for m in models]
    names = list(models.keys())
    bars  = ax.bar(names, vals, color=color, alpha=0.85,
                   edgecolor="black", linewidth=0.8)
    ax.set_title(f"{metric}\n{subtitle}", fontsize=10, fontweight="bold")
    ax.set_ylabel(metric, fontsize=10)
    ax.set_ylim(0, max(vals) * 1.3)
    ax.tick_params(axis="x", rotation=15)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(vals)*0.02,
                f"{val:.3f}", ha="center", va="bottom",
                fontsize=10, fontweight="bold")
    ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("Biomass Estimation Model Performance\n"
             "Features: Vegetation Indices + CHM + Volume  |  "
             "Target: Actual Biomass (840nm)  |  "
             "70% Training | 30% Testing",
             fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, "plot1_estimation_metrics.png"),
            dpi=100, bbox_inches="tight", facecolor="white")
plt.show()
plt.close("all")
print("   ✅ Saved: plot1_estimation_metrics.png")



🖼️  Plot 1: Estimation metrics comparison...
   ✅ Saved: plot1_estimation_metrics.png


In [67]:
# 14. PLOT 2 — ESTIMATED vs ACTUAL BIOMASS (840nm)
# ============================================================
print("🖼️  Plot 2: Estimated vs Actual biomass ...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=80)

for ax, (name, y_estimated) in zip(axes, estimations.items()):
    r2   = results[name]["R²"]
    rmse = results[name]["RMSE"]
    mn   = min(y_test.min(), y_estimated.min())
    mx   = max(y_test.max(), y_estimated.max())

    ax.scatter(y_test, y_estimated, alpha=0.7, edgecolors="black",
               linewidth=0.4, color="#1976D2", s=60,
               label="Estimated vs Actual")
    ax.plot([mn, mx], [mn, mx], "r--", linewidth=1.8, label="1:1 line")
    z = np.polyfit(y_test, y_estimated, 1)
    ax.plot(sorted(y_test), np.poly1d(z)(sorted(y_test)),
            "g-", linewidth=1.5, label="Estimation trend")

    ax.set_xlabel("Actual Biomass (g/plant)", fontsize=10)
    ax.set_ylabel("Estimated Biomass (g/plant)", fontsize=10)
    ax.set_title(f"{name}\nR² = {r2:.4f} | RMSE = {rmse:.4f}",
                 fontsize=10, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("Estimated vs Actual Biomass — Measurements (30% Test Set)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, "plot2_estimated_vs_actual_biomass.png"),
            dpi=100, bbox_inches="tight", facecolor="white")
plt.show()
plt.close("all")
print("   ✅ Saved: plot2_estimated_vs_actual_biomass.png")


🖼️  Plot 2: Estimated vs Actual biomass ...
   ✅ Saved: plot2_estimated_vs_actual_biomass.png


In [68]:
# 15. PLOT 3 — RESIDUALS SCATTER
# ============================================================
print("🖼️  Plot 3: Residuals scatter...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=80)

for ax, (name, y_estimated) in zip(axes, estimations.items()):
    residuals = y_test - y_estimated
    ax.scatter(y_estimated, residuals, alpha=0.7, edgecolors="black",
               linewidth=0.4, color="#7B1FA2", s=60)
    ax.axhline(0, color="red", linestyle="--", linewidth=1.8)
    ax.set_xlabel("Estimated Biomass", fontsize=11)
    ax.set_ylabel("Residuals\n(Actual − Estimated)", fontsize=10)
    ax.set_title(f"{name}\nResiduals", fontsize=11, fontweight="bold")
    ax.grid(True, alpha=0.3)

plt.suptitle("Biomass Estimation Residual Analysis\n"
             "(Actual Biomass − Estimated Biomass)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, "plot3_estimation_residuals.png"),
            dpi=100, bbox_inches="tight", facecolor="white")
plt.show()
plt.close("all")
print("   ✅ Saved: plot3_estimation_residuals.png")


🖼️  Plot 3: Residuals scatter...
   ✅ Saved: plot3_estimation_residuals.png


In [69]:
# 16. PLOT 4 — RESIDUAL DISTRIBUTION
# ============================================================
print("🖼️  Plot 4: Residual distribution...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=80)

for ax, (name, y_estimated) in zip(axes, estimations.items()):
    residuals = y_test - y_estimated
    ax.hist(residuals, bins=20, color="#00796B",
            edgecolor="black", alpha=0.8, linewidth=0.6)
    ax.axvline(0, color="red", linestyle="--", linewidth=1.8)
    ax.axvline(np.mean(residuals), color="blue", linestyle="-",
               linewidth=1.5, label=f"Mean = {np.mean(residuals):.3f}")
    ax.set_xlabel("Residual (Actual − Estimated)", fontsize=10)
    ax.set_ylabel("Frequency", fontsize=11)
    ax.set_title(f"{name}\nResidual Distribution", fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("Biomass Estimation Residual Distributions",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, "plot4_residual_distribution.png"),
            dpi=100, bbox_inches="tight", facecolor="white")
plt.show()
plt.close("all")
print("   ✅ Saved: plot4_residual_distribution.png")


🖼️  Plot 4: Residual distribution...
   ✅ Saved: plot4_residual_distribution.png


In [70]:
# 17. PLOT 5 — CROSS VALIDATION R²
# ============================================================
print("🖼️  Plot 5: Cross-validation R²...")

fig, ax = plt.subplots(figsize=(9, 5), dpi=80)
cv_data  = [cv_scores[m] for m in models]
cv_names = list(models.keys())

bp = ax.boxplot(cv_data, patch_artist=True, notch=False,
                medianprops=dict(color="red", linewidth=2))
for patch, color in zip(bp["boxes"], ["#2196F3", "#4CAF50", "#FF5722"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_xticklabels(cv_names, fontsize=11)
ax.set_ylabel("R² — Coefficient of Determination", fontsize=11)
ax.set_title("5-Fold Cross Validation R²\n"
             "(Evaluated on 70% Training Set | Target: Actual Biomass)",
             fontsize=11, fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, "plot5_cross_validation_r2.png"),
            dpi=100, bbox_inches="tight", facecolor="white")
plt.show()
plt.close("all")
print("   ✅ Saved: plot5_cross_validation_r2.png")



🖼️  Plot 5: Cross-validation R²...
   ✅ Saved: plot5_cross_validation_r2.png


In [71]:
# ============================================================
# 18. PLOT 6 — FEATURE IMPORTANCE (RF & XGB)
# ============================================================
print("🖼️  Plot 6: Feature importance...")

from matplotlib.patches import Patch

# 🎨 Color mapping by feature type
source_colors = {
    # 🌿 Vegetation Indices (means)
    "ndvi_mean"  : "#42A5F5",
    "gndvi_mean" : "#42A5F5",
    "ndre_mean"  : "#42A5F5",
    "msavi_mean" : "#42A5F5",
    "evi_mean"   : "#42A5F5",
    "osavi_mean" : "#42A5F5",
    "vari_mean"  : "#42A5F5",
    "arvi_mean"  : "#42A5F5",

    # 🌿 Vegetation variability (std)
    "ndvi_std"   : "#1E88E5",
    "ndre_std"   : "#1E88E5",
    "evi_std"    : "#1E88E5",

    # 🌳 Structure (CHM + canopy)
    "canopy_cover": "#66BB6A",
    "chm_max"     : "#66BB6A",
    "chm_mean"    : "#66BB6A",
    "chm_std"     : "#66BB6A",

    # 📦 Volume
    "volume"      : "#FFA726",
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=100)

for ax, name in zip(axes, ["Random Forest", "XGBoost"]):
    importance = models[name].feature_importances_

    # Sort importance
    idx = np.argsort(importance)
    sorted_features = [FEATURES[i] for i in idx]
    sorted_importance = importance[idx]

    # Assign colors safely
    bar_colors = [source_colors.get(f.lower(), "#9E9E9E") for f in sorted_features]

    bars = ax.barh(
        sorted_features,
        sorted_importance,
        color=bar_colors,
        alpha=0.85,
        edgecolor="black",
        linewidth=0.6
    )

    ax.set_xlabel("Importance Score", fontsize=11)
    ax.set_title(
        f"{name}\nFeature Importance → Biomass Estimation",
        fontsize=10,
        fontweight="bold"
    )
    ax.grid(True, alpha=0.3, axis="x")

    # Add values
    for bar, val in zip(bars, sorted_importance):
        ax.text(
            val + 0.002,
            bar.get_y() + bar.get_height()/2,
            f"{val:.3f}",
            va="center",
            fontsize=8
        )

# 🎨 Legend
legend_elements = [
    Patch(facecolor="#42A5F5", label="Vegetation Indices (Mean)"),
    Patch(facecolor="#1E88E5", label="Vegetation Indices (Std)"),
    Patch(facecolor="#66BB6A", label="Structure (CHM / Canopy)"),
    Patch(facecolor="#FFA726", label="Volume"),
]

axes[1].legend(handles=legend_elements, loc="lower right", fontsize=9)

plt.suptitle(
    "Feature Importance for Biomass Estimation",
    fontsize=13,
    fontweight="bold"
)

plt.tight_layout()

# Save plot
plot_path = os.path.join(OUTPUT_PATH, "plot6_feature_importance.png")
plt.savefig(plot_path, dpi=120, bbox_inches="tight", facecolor="white")

plt.show()
plt.close("all")

print(f"   ✅ Saved: {plot_path}")

🖼️  Plot 6: Feature importance...
   ✅ Saved: /content/drive/MyDrive/Biomass_Estimation_Project/01_RawData/data/Biomass_Estimation_Outputs/plot6_feature_importance.png


In [72]:
# 19. PLOT 7 — CORRELATION HEATMAP
# ============================================================
print("🖼️  Plot 7: Correlation heatmap...")

fig, ax = plt.subplots(figsize=(10, 8), dpi=80)
corr = df[FEATURES + [TARGET]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

# Rename target label for display
corr_display = corr.copy()
corr_display.columns = [c if c != TARGET else "Actual Biomass \n)" for c in corr.columns]
corr_display.index   = [c if c != TARGET else "Actual Biomass\n)" for c in corr.index]

sns.heatmap(corr_display, annot=True, fmt=".2f", cmap="RdYlGn",
            mask=mask, ax=ax, linewidths=0.5,
            annot_kws={"size": 10}, vmin=-1, vmax=1)
ax.set_title("Feature Correlation with Actual Biomass \n"
             "Vegetation Indices + CHM + Volume",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, "plot7_correlation_heatmap.png"),
            dpi=100, bbox_inches="tight", facecolor="white")
plt.show()
plt.close("all")
print("   ✅ Saved: plot7_correlation_heatmap.png")



🖼️  Plot 7: Correlation heatmap...
   ✅ Saved: plot7_correlation_heatmap.png


In [73]:
# 20. PLOT 8 — SHAP VALUES (XGBoost)
# ============================================================
print("🖼️  Plot 8: SHAP values — XGBoost biomass estimation...")

explainer   = shap.Explainer(models["XGBoost"], X_train)
shap_values = explainer(X_test)

fig, ax = plt.subplots(figsize=(10, 6), dpi=80)
shap.summary_plot(shap_values, X_test,
                  feature_names=FEATURES,
                  show=False, plot_size=None)
plt.title("SHAP Feature Contribution — XGBoost\nEstimating Actual Biomass",
          fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, "plot8_shap_xgboost.png"),
            dpi=100, bbox_inches="tight", facecolor="white")
plt.show()
plt.close("all")
del explainer, shap_values
gc.collect()
print("   ✅ Saved: plot8_shap_xgboost.png")


🖼️  Plot 8: SHAP values — XGBoost biomass estimation...
   ✅ Saved: plot8_shap_xgboost.png


In [74]:
# 21. PLOT 9 — RMSE & MAE COMPARISON
# ============================================================
print("🖼️  Plot 9: RMSE and MAE comparison...")

model_names = list(models.keys())
rmse_vals   = [results[m]["RMSE"] for m in model_names]
mae_vals    = [results[m]["MAE"]  for m in model_names]
x           = np.arange(len(model_names))
width       = 0.35

fig, ax = plt.subplots(figsize=(10, 5), dpi=80)
b1 = ax.bar(x - width/2, rmse_vals, width, label="RMSE",
            color="#E53935", alpha=0.85, edgecolor="black", linewidth=0.7)
b2 = ax.bar(x + width/2, mae_vals,  width, label="MAE",
            color="#FB8C00", alpha=0.85, edgecolor="black", linewidth=0.7)

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylabel("Error Value", fontsize=11)
ax.set_title("RMSE vs MAE — Biomass Estimation Models\n"
             "Target: Actual Biomass | lower = better estimation accuracy",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis="y")
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.002,
            f"{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, "plot9_rmse_mae_comparison.png"),
            dpi=100, bbox_inches="tight", facecolor="white")
plt.show()
plt.close("all")
print("   ✅ Saved: plot9_rmse_mae_comparison.png")


🖼️  Plot 9: RMSE and MAE comparison...
   ✅ Saved: plot9_rmse_mae_comparison.png


In [75]:
# ============================================================
# 22. PLOT 10 — FEATURE SOURCE CONTRIBUTION PIE CHART
# ============================================================
print("🖼️  Plot 10: Feature source contribution...")

# ============================================================
# 🌿 DEFINE FEATURE GROUPS (UPDATED)
# ============================================================

FEATURES_INDICES_MEAN = [
    "ndvi_mean", "gndvi_mean", "ndre_mean",
    "msavi_mean", "evi_mean", "osavi_mean",
    "vari_mean", "arvi_mean"
]

FEATURES_INDICES_STD = [
    "ndvi_std", "ndre_std", "evi_std"
]

FEATURES_STRUCTURE = [
    "canopy_cover", "chm_max", "chm_mean", "chm_std"
]

FEATURES_VOLUME = [
    "volume"
]

# Group dictionary
source_groups = {
    "Vegetation Indices (Mean)" : FEATURES_INDICES_MEAN,
    "Vegetation Indices (Std)"  : FEATURES_INDICES_STD,
    "Structure (CHM/Canopy)"    : FEATURES_STRUCTURE,
    "Volume"                   : FEATURES_VOLUME,
}

# ============================================================
# 🎨 COLORS
# ============================================================
group_colors = [
    "#42A5F5",  # mean indices
    "#1E88E5",  # std indices
    "#66BB6A",  # structure
    "#FFA726",  # volume
]

# ============================================================
# 📊 PLOT
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

for ax, name in zip(axes, ["Random Forest", "XGBoost"]):

    importance = models[name].feature_importances_
    feat_imp   = dict(zip(FEATURES, importance))

    group_names = list(source_groups.keys())

    # Safe summation (avoid missing feature errors)
    group_vals = [
        sum(feat_imp.get(f, 0) for f in feats)
        for feats in source_groups.values()
    ]

    wedges, texts, autotexts = ax.pie(
        group_vals,
        labels=group_names,
        autopct="%1.1f%%",
        colors=group_colors,
        startangle=90,
        wedgeprops=dict(edgecolor="white", linewidth=2)
    )

    for text in autotexts:
        text.set_fontsize(10)
        text.set_fontweight("bold")

    ax.set_title(
        f"{name}\nFeature Source Contribution",
        fontsize=10,
        fontweight="bold"
    )

# ============================================================
# 🏷️ TITLE & SAVE
# ============================================================
plt.suptitle(
    "Data Source Contribution to Biomass Estimation",
    fontsize=13,
    fontweight="bold"
)

plt.tight_layout()

plot_path = os.path.join(
    OUTPUT_PATH,
    "plot10_feature_source_contribution.png"
)

plt.savefig(plot_path, dpi=120, bbox_inches="tight", facecolor="white")

plt.show()
plt.close("all")

print(f"   ✅ Saved: {plot_path}")

🖼️  Plot 10: Feature source contribution...
   ✅ Saved: /content/drive/MyDrive/Biomass_Estimation_Project/01_RawData/data/Biomass_Estimation_Outputs/plot10_feature_source_contribution.png


In [76]:
# ============================================================
# 23. FINAL SUMMARY
# ============================================================

best_model = metrics_df["R²"].idxmax()

print("\n" + "="*65)
print("✅ BIOMASS ESTIMATION PIPELINE — COMPLETE")
print("="*65)

# ============================================================
# 📂 DATA SETUP
# ============================================================
print(f"\n   ── Data Setup ────────────────────────────────────────")
print(f"   Dataset         : Combined dataset (single file)")
print(f"   Features ({len(FEATURES)}) : {FEATURES}")
print(f"   Target          : {TARGET} (Actual Biomass)")

# Categorize features for clarity
FEATURES_INDICES_MEAN = [
    "ndvi_mean", "gndvi_mean", "ndre_mean",
    "msavi_mean", "evi_mean", "osavi_mean",
    "vari_mean", "arvi_mean"
]

FEATURES_INDICES_STD = [
    "ndvi_std", "ndre_std", "evi_std"
]

FEATURES_STRUCTURE = [
    "canopy_cover", "chm_max", "chm_mean", "chm_std"
]

FEATURES_VOLUME = ["volume"]

print(f"\n   Feature Groups:")
print(f"   🌿 Vegetation Indices (Mean) : {len([f for f in FEATURES if f in FEATURES_INDICES_MEAN])}")
print(f"   🌿 Vegetation Indices (Std)  : {len([f for f in FEATURES if f in FEATURES_INDICES_STD])}")
print(f"   🌳 Structure (CHM/Canopy)    : {len([f for f in FEATURES if f in FEATURES_STRUCTURE])}")
print(f"   📦 Volume                    : {len([f for f in FEATURES if f in FEATURES_VOLUME])}")

# ============================================================
# 🤖 TRAINING SETUP
# ============================================================
print(f"\n   ── Training Setup ────────────────────────────────────")
print(f"   Split            : 70% Training | 30% Testing")
print(f"   Total samples    : {len(X)}")
print(f"   Training         : {len(X_train)} samples (70%)")
print(f"   Testing          : {len(X_test)} samples (30%)")

# ============================================================
# 📊 MODEL RESULTS
# ============================================================
print(f"\n📊 Biomass Estimation Metrics:\n{metrics_df.to_string()}")

print(f"\n🏆 Best Estimation Model (R²) : {best_model}")
print(f"   R²   : {metrics_df.loc[best_model, 'R²']:.4f}")
print(f"   RMSE : {metrics_df.loc[best_model, 'RMSE']:.4f}")
print(f"   MAE  : {metrics_df.loc[best_model, 'MAE']:.4f}")

# ============================================================
# 💾 OUTPUTS
# ============================================================
print(f"\n📁 All outputs saved to : {OUTPUT_PATH}")

# Updated dataset saving (no merge anymore)
final_dataset_path = os.path.join(OUTPUT_PATH, "final_biomass_dataset.csv")
print(f"💾 Final dataset saved : {final_dataset_path}")

# ============================================================
# 📈 GENERATED PLOTS
# ============================================================
print("\n📈 Plots generated:")

plots = [
    "plot1_estimation_metrics.png",
    "plot2_estimated_vs_actual_biomass.png",
    "plot3_estimation_residuals.png",
    "plot4_residual_distribution.png",
    "plot5_cross_validation_r2.png",
    "plot6_feature_importance.png",
    "plot7_correlation_heatmap.png",
    "plot8_shap_xgboost.png",
    "plot9_rmse_mae_comparison.png",
    "plot10_feature_source_contribution.png",
    "biomass_estimation_metrics.csv",
    "final_biomass_dataset.csv",  # ✅ updated
]

for p in plots:
    full   = os.path.join(OUTPUT_PATH, p)
    size   = os.path.getsize(full) / 1024 if os.path.exists(full) else 0
    status = f"✅ {size:.1f} KB" if size > 5 else "⚠️ may be blank"
    print(f"   {status} — {p}")


✅ BIOMASS ESTIMATION PIPELINE — COMPLETE

   ── Data Setup ────────────────────────────────────────
   Dataset         : Combined dataset (single file)
   Features (10) : ['volume', 'chm_max', 'chm_mean', 'canopy_cover', 'vari_mean', 'gndvi_mean', 'ndvi_mean', 'ndvi_std', 'ndre_std', 'chm_std']
   Target          : actual_biomass (Actual Biomass)

   Feature Groups:
   🌿 Vegetation Indices (Mean) : 3
   🌿 Vegetation Indices (Std)  : 2
   🌳 Structure (CHM/Canopy)    : 4
   📦 Volume                    : 1

   ── Training Setup ────────────────────────────────────
   Split            : 70% Training | 30% Testing
   Total samples    : 708
   Training         : 495 samples (70%)
   Testing          : 213 samples (30%)

📊 Biomass Estimation Metrics:
                                  R²     RMSE      MAE  CV_R²_mean  CV_R²_std
Estimation Model                                                             
Support Vector Machine (SVM)  0.5248  26.0398  20.6443      0.2478     0.0896
Random Fore